# IR spectra agent on Colab (UMA on GPU)

Runs the validation matrix with **Meta UMA** (gated) on a Colab GPU. Results go to Google Drive and the
run **resumes** after a disconnect: every finished case writes `row.json` (each case < 10 min, so the
checkpoint interval is well under 20 min). Colab has no background execution, so keep the tab open.

Before running: (1) set `REPO_URL`/`BRANCH` below if you use a fork or another branch; (2) request access on
https://huggingface.co/facebook/UMA and create a read token (gated repos).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/ir_agent'          # outputs + resume state live here
REPO_URL = 'https://github.com/ryanlai666/IR_spectra_agent.git'
BRANCH = 'main'

In [ ]:
import os, subprocess
os.makedirs(DRIVE_ROOT, exist_ok=True)
if not os.path.exists('/content/IR_spectra_agent'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, '/content/IR_spectra_agent'], check=True)
%cd /content/IR_spectra_agent
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Install
fairchem-core 2.21.0 pins torch 2.8 (pip will install it). `tblite` has Linux wheels, giving GFN2-xTB
dipoles in-process; the xtb binary comes from the official release for the GFN2 cluster runs.

In [ ]:
!pip -q install -e ".[uma,llm,dev]" tblite
!curl -fsSL -o /tmp/xtb.tar.xz https://github.com/grimme-lab/xtb/releases/download/v6.7.1/xtb-6.7.1-linux-x86_64.tar.xz
!mkdir -p .tools/xtb-release && tar -xf /tmp/xtb.tar.xz -C .tools/xtb-release
import glob; os.environ['IR_AGENT_XTB_BIN'] = [p for p in glob.glob('.tools/xtb-release/**/bin/xtb', recursive=True)][0]
os.environ['IR_AGENT_DEDICATED_GPU'] = '1'   # Colab GPU is not shared: allow the GPU-marked test
os.environ['MLFLOW_DISABLE_AGENT_HINT'] = '1'
!ir-agent calculators

## Hugging Face login + UMA-S 1.2.1 checkpoint (gated; your own access, nothing stored in the repo)
The `uma` preset loads `data/checkpoints/uma-s-1p2p1.pt` locally (fairchem `load_predict_unit`) and
refuses to run if its MD5 is not `3497615fd30a24c5b35cd3b41a682e6e`.

In [ ]:
from huggingface_hub import notebook_login, hf_hub_download
import hashlib, shutil, pathlib
notebook_login()
src = hf_hub_download("facebook/UMA", "uma-s-1p2p1.pt", subfolder="checkpoints")
dst = pathlib.Path("data/checkpoints/uma-s-1p2p1.pt"); dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(src, dst)
h = hashlib.md5(); f = open(dst, "rb")
for b in iter(lambda: f.read(8 << 20), b""): h.update(b)
assert h.hexdigest() == "3497615fd30a24c5b35cd3b41a682e6e", h.hexdigest()
print("uma-s-1p2p1.pt OK", dst.stat().st_size)

## GPU smoke test: real UMA-S, H2O hybrid IR (records wall time + peak VRAM)

In [ ]:
!pytest -q -m "gpu and uma" tests/test_uma.py -s

## Validation matrix with resume
Re-running this cell after a disconnect continues where it stopped (`--resume`).

In [ ]:
import pathlib
RESUME = pathlib.Path(DRIVE_ROOT) / 'validation_uma'
RESUME.mkdir(parents=True, exist_ok=True)
for sub in ('structures', 'calc', 'plots'):
    (RESUME / sub).mkdir(exist_ok=True)
!ir-agent validate --calculators uma-omol,uma-oc20,uma-omat,xtb-gfn1 --device cuda --resume {RESUME}

In [ ]:
from IPython.display import Markdown, display
display(Markdown((RESUME / 'validation.md').read_text()))

## Optional: one agentic run with the Gemini planner
Needs `GOOGLE_API_KEY` (Colab secrets). Without it use `--no-llm --plan examples/plans/co_pt111_uma.json`.

In [ ]:
# from google.colab import userdata; os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
!ir-agent run --no-llm --plan examples/plans/co_pt111_uma.json --output-root {DRIVE_ROOT}/runs